# LeetCode 2101 — Detonate the Maximum Bombs bằng Python

Link đề: https://leetcode.com/problems/detonate-the-maximum-bombs/ · Độ khó: **Medium** · Chủ đề: Array, Math, Depth-First Search, Breadth-First Search, Graph, Geometry

Skeleton LeetCode:

```python
class Solution:
    def maximumDetonation(self, bombs: List[List[int]]) -> int:
        ...
```

Lộ trình:
1. **Mô phỏng thuần**: kích nổ một quả, lặp quét mọi quả bom cho tới khi không còn quả mới nổ — `O(n³)` mỗi lần thử, `O(n⁴)` tổng.
2. **Đồ thị có hướng + BFS từ mỗi đỉnh** — `O(n²)` dựng cạnh, `O(n³)` tổng (**nộp bài**).
3. **DFS lặp + dừng sớm** khi đã nổ đủ `n` quả.
4. **Bao đóng bắc cầu bằng bitset** (Floyd–Warshall trên `int` Python) — `O(n³ / 64)`.

## 1. Mục tiêu học

1. Mô hình hóa "bom `i` kích nổ bom `j`" thành **cạnh có hướng** `i → j` khi `j` nằm trong (hoặc trên) đường tròn của `i` — quan hệ **không đối xứng**.
2. So sánh khoảng cách bằng **bình phương số nguyên** (`dx² + dy² ≤ r²`) để tránh sai số số thực.
3. Đếm số đỉnh **đến được** từ mỗi đỉnh bằng BFS/DFS và lấy max.
4. Hiểu vì sao Union-Find (đồ thị vô hướng) **sai** ở bài này.
5. Biết kỹ thuật bao đóng bắc cầu bằng bitset — hữu ích khi cần "tập đến được" của mọi đỉnh.

## 2. Đề bài & ví dụ

Mỗi quả bom `bombs[i] = [x, y, r]` có tâm `(x, y)` và bán kính ảnh hưởng `r`. Khi một quả nổ, mọi quả bom nằm **trong** vùng tròn của nó (kể cả trên biên) cũng nổ, và tiếp tục lan. Được chọn **một** quả để kích nổ; trả về số bom **nhiều nhất** có thể nổ.

**Ví dụ 1:** `[[2,1,3],[6,1,4]]` → `2` (nổ quả phải thì lan sang quả trái; ngược lại không).

**Ví dụ 2:** `[[1,1,5],[10,10,5]]` → `1`.

**Ví dụ 3:** `[[1,2,3],[2,3,1],[3,4,2],[4,5,3],[5,6,4]]` → `5` (bom 0 → 1, 2; bom 2 → 3; bom 3 → 4).

**Ràng buộc:** `1 ≤ n ≤ 100`, `1 ≤ x, y, r ≤ 10⁵`.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [([[2, 1, 3], [6, 1, 4]], 2), ([[1, 1, 5], [10, 10, 5]], 1),
            ([[1, 2, 3], [2, 3, 1], [3, 4, 2], [4, 5, 3], [5, 6, 4]], 5)]
for bombs, want in examples:
    print(bombs, "->", want)

[[2, 1, 3], [6, 1, 4]] -> 2
[[1, 1, 5], [10, 10, 5]] -> 1
[[1, 2, 3], [2, 3, 1], [3, 4, 2], [4, 5, 3], [5, 6, 4]] -> 5


## 3. Kiến thức nền

### 3.1 Điểm trong đường tròn — so sánh bằng bình phương

`(xj, yj)` trong đường tròn tâm `(xi, yi)` bán kính `ri` ⇔ `(xi − xj)² + (yi − yj)² ≤ ri²`. Toàn số nguyên ⇒ chính xác tuyệt đối (Python không tràn số).

In [2]:
def reaches(a, b):
    """Bom a kích nổ được bom b?"""
    dx, dy = a[0] - b[0], a[1] - b[1]
    return dx * dx + dy * dy <= a[2] * a[2]

A, B = [2, 1, 3], [6, 1, 4]
print("A -> B:", reaches(A, B), "| B -> A:", reaches(B, A), "(khoảng cách 4: chỉ B với r = 4 với tới)")

A -> B: False | B -> A: True (khoảng cách 4: chỉ B với r = 4 với tới)


### 3.2 Danh sách kề có hướng

In [3]:
def build_graph(bombs):
    n = len(bombs)
    return [[j for j in range(n) if j != i and reaches(bombs[i], bombs[j])] for i in range(n)]

for i, nb_ in enumerate(build_graph(examples[2][0])):
    print(i, "->", nb_)

0 -> [1, 2]
1 -> []
2 -> [1, 3]
3 -> [1, 2, 4]
4 -> [2, 3]


### 3.3 BFS đếm số đỉnh đến được

In [4]:
def count_reachable(g, s):
    seen = [False] * len(g); seen[s] = True; q = deque([s]); cnt = 1
    while q:
        u = q.popleft()
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; cnt += 1; q.append(v)
    return cnt

g3 = build_graph(examples[2][0])
print([count_reachable(g3, s) for s in range(len(g3))])

[5, 1, 4, 4, 4]


### 3.4 Bitset bằng `int`

Tập con của `{0..n−1}` lưu bằng số nguyên: bit `j` bật nếu `j` thuộc tập. Hợp hai tập là `|`, đếm phần tử là `.bit_count()`.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3 — đồ thị có hướng:

```
0 -> 1, 2          (bom 0 bán kính 3: d² tới 1 là 2, tới 2 là 8 ≤ 9)
1 -> (không)       (bán kính 1, mọi bom khác có d² ≥ 2)
2 -> 1, 3
3 -> 1, 2, 4
4 -> 2, 3
```

Từ `0`: `0 → {1, 2}`, `2 → {3}`, `3 → {4}` ⇒ cả 5 quả. Từ `4`: `4 → {2, 3}`, `2 → {1}` ⇒ `{4, 2, 3, 1}` — **không** về được `0` vì không có cạnh nào đi vào `0`. Từ `1`: chỉ chính nó.

**Vì sao phải có hướng?** Ví dụ 1: khoảng cách giữa hai tâm là `4`; bom phải (`r = 4`) với tới bom trái, bom trái (`r = 3`) **không** với tới bom phải. Quan hệ "kích nổ" không đối xứng.

In [5]:
for s in range(5):
    seen = {s}; q = deque([s]); order = [s]
    while q:
        u = q.popleft()
        for v in g3[u]:
            if v not in seen: seen.add(v); q.append(v); order.append(v)
    print(f"kích nổ bom {s}: thứ tự nổ {order} -> {len(order)} quả")

kích nổ bom 0: thứ tự nổ [0, 1, 2, 3, 4] -> 5 quả
kích nổ bom 1: thứ tự nổ [1] -> 1 quả
kích nổ bom 2: thứ tự nổ [2, 1, 3, 4] -> 4 quả
kích nổ bom 3: thứ tự nổ [3, 1, 2, 4] -> 4 quả
kích nổ bom 4: thứ tự nổ [4, 2, 3, 1] -> 4 quả


## 5. Cách 1 — Brute Force / baseline: mô phỏng thuần

**Ý tưởng:** không dựng đồ thị. Với mỗi bom bắt đầu, giữ tập `exploded`; lặp: quét mọi cặp (bom đã nổ, bom chưa nổ), nếu với tới thì cho nổ; dừng khi một vòng không có bom mới.

In [6]:
def max_bombs_simulate(bombs):
    n = len(bombs); best = 0
    for s in range(n):
        exploded = [False] * n; exploded[s] = True
        changed = True
        while changed:
            changed = False
            for i in range(n):
                if exploded[i]:
                    for j in range(n):
                        if not exploded[j] and reaches(bombs[i], bombs[j]):
                            exploded[j] = True; changed = True
        best = max(best, sum(exploded))
    return best

for bombs, want in examples:
    print(max_bombs_simulate(bombs), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** mỗi vòng `O(n²)`, tối đa `n` vòng mỗi bom bắt đầu ⇒ `O(n³)` mỗi bom, `O(n⁴) = 10⁸` tệ nhất với `n = 100` — chậm trong Python. Space `O(n)`.

**Nút thắt:** tính lại `reaches(i, j)` nhiều lần và quét lại những bom đã xử lý.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** quan hệ "kích nổ trực tiếp" chỉ phụ thuộc cặp `(i, j)` ⇒ tính **một lần** thành đồ thị có hướng `n` đỉnh. Số bom nổ khi kích bom `s` = số đỉnh **đến được** từ `s` (tính cả `s`). Đáp án = `max_s |reach(s)|`.

**Quy tắc:**
- *Nếu* `dx² + dy² ≤ rᵢ²` *thì* có cạnh `i → j` (dùng `≤` vì "trong vùng" gồm cả biên).
- Mỗi lần BFS dùng mảng `seen` **mới**.
- *Nếu* một lần BFS đạt `n` *thì* dừng sớm — không thể tốt hơn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đồ thị có hướng + BFS từ mỗi đỉnh (code nộp)

In [7]:
def max_bombs_bfs(bombs):
    g = build_graph(bombs)
    return max(count_reachable(g, s) for s in range(len(bombs)))

for bombs, want in examples:
    print(max_bombs_bfs(bombs), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
5 kỳ vọng 5


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** dựng đồ thị `O(n²)`; mỗi BFS `O(n + E)` với `E ≤ n²` ⇒ tổng `O(n · (n + E)) = O(n³) = 10⁶`. Space `O(n²)`.

### 7.2 Cách 3 — DFS lặp + dừng sớm

In [8]:
def max_bombs_dfs(bombs):
    n = len(bombs); g = build_graph(bombs); best = 0
    for s in range(n):
        seen = [False] * n; seen[s] = True; st = [s]; cnt = 1
        while st:
            u = st.pop()
            for v in g[u]:
                if not seen[v]:
                    seen[v] = True; cnt += 1; st.append(v)
        best = max(best, cnt)
        if best == n:
            break                                   # đã nổ hết — không thể tốt hơn
    return best

for bombs, want in examples:
    print(max_bombs_dfs(bombs), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** như Cách 2 trong trường hợp xấu nhất; dừng sớm giúp nhiều khi có một quả "lan" hết.

### 7.3 Cách 4 — Bao đóng bắc cầu bằng bitset

`reach[i]` ban đầu là bit của `i` và các láng giềng trực tiếp. Duyệt `k = 0..n−1`: với mọi `i` mà `k ∈ reach[i]` thì `reach[i] |= reach[k]`. Đây là Floyd–Warshall cho bài toán "đến được", mỗi phép OR xử lý 64 bit một lúc.

In [9]:
def max_bombs_bitset(bombs):
    n = len(bombs)
    reach = [1 << i for i in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j and reaches(bombs[i], bombs[j]):
                reach[i] |= 1 << j
    for k in range(n):
        bit = 1 << k; rk = reach[k]
        for i in range(n):
            if reach[i] & bit:
                reach[i] |= rk
    return max(r.bit_count() for r in reach)

for bombs, want in examples:
    print(max_bombs_bitset(bombs), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
5 kỳ vọng 5


**Độ phức tạp:** `O(n²)` dựng + `O(n²)` phép OR trên số `n` bit ⇒ `O(n³ / 64)`. Space `O(n² / 64)`. Bonus: ta có **toàn bộ** tập đến được của mọi đỉnh, không chỉ số lượng.

**Vì sao đúng?** Bất biến sau bước `k`: `reach[i]` chứa mọi `j` đến được từ `i` bằng đường đi mà các đỉnh **trung gian** đều thuộc `{0..k}` — đúng lập luận Floyd–Warshall.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 3: danh sách cạnh kèm khoảng cách², rồi BFS từ bom 0.

In [10]:
def trace_edges(bombs):
    n = len(bombs)
    for i in range(n):
        for j in range(n):
            if i != j:
                dx, dy = bombs[i][0] - bombs[j][0], bombs[i][1] - bombs[j][1]
                d2, r2 = dx * dx + dy * dy, bombs[i][2] ** 2
                if d2 <= r2:
                    print(f"  {i} -> {j}: d² = {d2} <= r² = {r2}")

def trace_bfs(bombs, s):
    g = build_graph(bombs); seen = {s}; q = deque([s]); step = 0
    print(f"BFS từ bom {s}:")
    while q:
        u = q.popleft(); new = [v for v in g[u] if v not in seen]
        for v in new: seen.add(v); q.append(v)
        step += 1
        print(f"  lấy {u}: nổ thêm {new}; hàng đợi {list(q)}; đã nổ {sorted(seen)}")
    return len(seen)

bombs3 = examples[2][0]
print("Các cạnh (ví dụ 3):"); trace_edges(bombs3)
trace_bfs(bombs3, 0)

Các cạnh (ví dụ 3):
  0 -> 1: d² = 2 <= r² = 9
  0 -> 2: d² = 8 <= r² = 9
  2 -> 1: d² = 2 <= r² = 4
  2 -> 3: d² = 2 <= r² = 4
  3 -> 1: d² = 8 <= r² = 9
  3 -> 2: d² = 2 <= r² = 9
  3 -> 4: d² = 2 <= r² = 9
  4 -> 2: d² = 8 <= r² = 16
  4 -> 3: d² = 2 <= r² = 16
BFS từ bom 0:
  lấy 0: nổ thêm [1, 2]; hàng đợi [1, 2]; đã nổ [0, 1, 2]
  lấy 1: nổ thêm []; hàng đợi [2]; đã nổ [0, 1, 2]
  lấy 2: nổ thêm [3]; hàng đợi [3]; đã nổ [0, 1, 2, 3]
  lấy 3: nổ thêm [4]; hàng đợi [4]; đã nổ [0, 1, 2, 3, 4]
  lấy 4: nổ thêm []; hàng đợi []; đã nổ [0, 1, 2, 3, 4]


5

In [11]:
def trace_bitset(bombs):
    n = len(bombs); reach = [1 << i for i in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j and reaches(bombs[i], bombs[j]): reach[i] |= 1 << j
    fmt = lambda r: "".join("1" if r >> j & 1 else "." for j in range(n))
    print("ban đầu:", [fmt(r) for r in reach])
    for k in range(n):
        for i in range(n):
            if reach[i] >> k & 1: reach[i] |= reach[k]
        print(f"sau k = {k}:", [fmt(r) for r in reach])

trace_bitset(bombs3)

ban đầu: ['111..', '.1...', '.111.', '.1111', '..111']
sau k = 0: ['111..', '.1...', '.111.', '.1111', '..111']
sau k = 1: ['111..', '.1...', '.111.', '.1111', '..111']
sau k = 2: ['1111.', '.1...', '.111.', '.1111', '.1111']
sau k = 3: ['11111', '.1...', '.1111', '.1111', '.1111']
sau k = 4: ['11111', '.1...', '.1111', '.1111', '.1111']


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def maximumDetonation(self, bombs: List[List[int]]) -> int:
        n = len(bombs)
        g = [[] for _ in range(n)]
        for i, (xi, yi, ri) in enumerate(bombs):
            r2 = ri * ri
            for j, (xj, yj, _) in enumerate(bombs):
                if i != j and (xi - xj) ** 2 + (yi - yj) ** 2 <= r2:
                    g[i].append(j)                      # i kích nổ j (có hướng)
        best = 0
        for s in range(n):
            seen = [False] * n
            seen[s] = True
            q = deque([s])
            cnt = 1
            while q:
                u = q.popleft()
                for v in g[u]:
                    if not seen[v]:
                        seen[v] = True
                        cnt += 1
                        q.append(v)
            best = max(best, cnt)
            if best == n:
                break
        return best

sol = Solution()
for bombs, want in examples:
    got = sol.maximumDetonation(bombs)
    print(got, "OK" if got == want else "SAI")

2 OK
1 OK
5 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (bom nổ = đỉnh đến được).** Kích bom `s`. Một bom `v` nổ ⇔ tồn tại dãy `s = u₀, u₁, …, u_k = v` mà `u_{t+1}` nằm trong vùng của `u_t` ⇔ có đường đi có hướng `s ⇝ v` trong đồ thị.
- (⇐) Quy nạp theo độ dài đường: `u_t` nổ ⇒ `u_{t+1}` trong vùng nên nổ.
- (⇒) Mỗi bom nổ do một bom đã nổ trước kích hoạt (hoặc là `s`) ⇒ truy ngược "ai kích nổ ai" ra một đường đi từ `s`.

BFS/DFS từ `s` thăm đúng tập đỉnh đến được ⇒ `cnt` = số bom nổ. Lấy max trên mọi `s` ⇒ đáp án. ✓

**Dừng sớm:** số bom nổ `≤ n`; đạt `n` thì không thể vượt. ✓

**Chính xác số học:** tọa độ và bán kính nguyên ⇒ so sánh bình phương nguyên là chính xác; dùng `sqrt` số thực có thể sai ở biên (Lỗi 2).

## 11. Phân tích Time / Space

`n ≤ 100`, `E ≤ n(n−1)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng thuần | `O(n⁴)` | `O(n)` |
| 2. Đồ thị + BFS | `O(n³)` | `O(n²)` |
| 3. DFS + dừng sớm | `O(n³)` tệ nhất | `O(n²)` |
| 4. Bitset closure | `O(n³ / 64)` + dựng `O(n²)` | `O(n² / 64)` |

In [13]:
rng = random.Random(1)
def random_bombs(n, rng, coord=200, rmax=60):
    return [[rng.randint(1, coord), rng.randint(1, coord), rng.randint(1, rmax)] for _ in range(n)]

tests = [("ngẫu nhiên thưa", random_bombs(100, rng, 10**5, 2000)),
         ("ngẫu nhiên dày", random_bombs(100, rng, 300, 60)),
         ("chuỗi một chiều", [[i * 10 + 1, 1, 10 if i % 2 == 0 else 9] for i in range(100)])]
for name, bombs in tests:
    res = []
    for fname, fn in [("simulate", max_bombs_simulate), ("bfs", max_bombs_bfs), ("dfs", max_bombs_dfs),
                      ("bitset", max_bombs_bitset), ("submit", Solution().maximumDetonation)]:
        t0 = time.perf_counter(); r = fn(bombs); res.append(r)
        print(f"{name:16s} {fname:8s}: {r:>3}  {(time.perf_counter() - t0) * 1000:.1f} ms")
    assert len(set(res)) == 1

ngẫu nhiên thưa  simulate:   2  1.9 ms
ngẫu nhiên thưa  bfs     :   2  1.5 ms
ngẫu nhiên thưa  dfs     :   2  1.5 ms
ngẫu nhiên thưa  bitset  :   2  1.8 ms
ngẫu nhiên thưa  submit  :   2  1.3 ms
ngẫu nhiên dày   simulate:  48  50.0 ms
ngẫu nhiên dày   bfs     :  48  1.5 ms
ngẫu nhiên dày   dfs     :  48  1.5 ms
ngẫu nhiên dày   bitset  :  48  1.7 ms
ngẫu nhiên dày   submit  :  48  1.2 ms
chuỗi một chiều  simulate:   3  3.2 ms
chuỗi một chiều  bfs     :   3  1.0 ms
chuỗi một chiều  dfs     :   3  1.1 ms
chuỗi một chiều  bitset  :   3  1.3 ms
chuỗi một chiều  submit  :   3  0.7 ms


## 12. Bộ test

In [14]:
fns = [max_bombs_bfs, max_bombs_dfs, max_bombs_bitset, Solution().maximumDetonation]

for bombs, want in examples:
    assert max_bombs_simulate(bombs) == want
    for fn in fns:
        assert fn(bombs) == want

edge = [
    ([[1, 1, 1]], 1),                                     # một quả
    ([[1, 1, 1], [1, 1, 1]], 2),                          # trùng tâm
    ([[1, 1, 3], [4, 1, 1]], 2),                          # đúng trên biên: d = 3 = r
    ([[1, 1, 2], [4, 1, 1]], 1),                          # d = 3 > 2
    ([[1, 1, 5], [3, 1, 1], [100, 100, 1]], 2),
    ([[1, 1, 3], [3, 1, 1], [5, 1, 3]], 2),               # 0 -> 1, 2 -> 1: không có "3"
    ([[1, 1, 5], [4, 5, 5], [7, 9, 5], [10, 13, 5]], 4),   # chuỗi với khoảng cách 5 = r (biên)
]
for bombs, want in edge:
    assert max_bombs_simulate(bombs) == want, bombs
    for fn in fns:
        assert fn(bombs) == want, (fn, bombs)

rng = random.Random(2024)
for _ in range(600):
    n = rng.randint(1, 15)
    bombs = random_bombs(n, rng, rng.choice([10, 30, 100]), rng.choice([3, 10, 30]))
    want = max_bombs_simulate(bombs)
    for fn in fns:
        assert fn(bombs) == want, (fn, bombs)
print("ví dụ + edge + 600 bộ ngẫu nhiên khớp mô phỏng thuần")

ví dụ + edge + 600 bộ ngẫu nhiên khớp mô phỏng thuần


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng đồ thị **vô hướng** / Union-Find

In [15]:
def wrong_union_find(bombs):
    n = len(bombs); p = list(range(n))
    def find(x):
        while p[x] != x: p[x] = p[p[x]]; x = p[x]
        return x
    for i in range(n):
        for j in range(n):
            if i != j and reaches(bombs[i], bombs[j]):
                p[find(i)] = find(j)                              # SAI: quan hệ không đối xứng
    from collections import Counter
    return max(Counter(find(i) for i in range(n)).values())

case = [[1, 1, 3], [3, 1, 1], [5, 1, 3]]                          # 0 -> 1 <- 2
for bombs, want in examples + [(case, 2)]:
    print(f"sai = {wrong_union_find(bombs)} | đúng = {want}")

sai = 2 | đúng = 2
sai = 1 | đúng = 1
sai = 5 | đúng = 5
sai = 3 | đúng = 2


Với `0 → 1 ← 2`, Union-Find gộp cả ba vào một nhóm ⇒ `3`, nhưng kích bất kỳ quả nào cũng chỉ nổ tối đa 2. **Sửa:** đồ thị **có hướng**, đếm đỉnh đến được.

### Lỗi 2 — So sánh khoảng cách bằng `sqrt` số thực / so sánh ngặt

In [16]:
import math

def wrong_strict(bombs):
    n = len(bombs)
    g = [[j for j in range(n) if j != i and
          (bombs[i][0] - bombs[j][0]) ** 2 + (bombs[i][1] - bombs[j][1]) ** 2 < bombs[i][2] ** 2]   # SAI: phải <=
         for i in range(n)]
    return max(count_reachable(g, s) for s in range(n))

def float_reaches(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) <= a[2]

print("trên biên (d = r = 3): sai(<) =", wrong_strict([[1, 1, 3], [4, 1, 1]]), "| đúng =", 2)
print("ví dụ 3 (tình cờ không có cạnh quyết định nằm đúng trên biên): sai(<) =", wrong_strict(examples[2][0]), "| đúng =", 5)
# sqrt có thể làm tròn sai với số lớn (ngoài ràng buộc bài, minh họa rủi ro)
a, b = [0, 0, 94906267], [94906267, 1, 1]
print("số lớn: d² =", 94906267 ** 2 + 1, "r² =", 94906267 ** 2, "| nguyên:", reaches(a, b), "| float sqrt:", float_reaches(a, b))

trên biên (d = r = 3): sai(<) = 1 | đúng = 2
ví dụ 3 (tình cờ không có cạnh quyết định nằm đúng trên biên): sai(<) = 5 | đúng = 5
số lớn: d² = 9007199515875290 r² = 9007199515875289 | nguyên: False | float sqrt: True


"Nằm trong vùng" gồm cả **biên** (cặp `[[1,1,3],[4,1,1]]`: khoảng cách đúng bằng `r`). Còn `sqrt` số thực có thể làm tròn khoảng cách hơi lớn hơn `r` thành đúng `r` (dòng cuối: `d² = r² + 1` mà `sqrt` vẫn cho `≤`). Trong ràng buộc bài (`≤ 10⁵`) sai số này khó xảy ra, nhưng thói quen an toàn là **so sánh bình phương nguyên**. **Sửa:** `dx² + dy² ≤ r²`.

### Lỗi 3 — Chỉ đếm láng giềng **trực tiếp** (quên lan truyền)

In [17]:
def wrong_direct_only(bombs):
    g = build_graph(bombs)
    return max(1 + len(g[i]) for i in range(len(bombs)))           # SAI: bom bị nổ cũng kích nổ tiếp

for bombs, want in examples:
    print(f"sai = {wrong_direct_only(bombs)} | đúng = {want}")

sai = 2 | đúng = 2
sai = 1 | đúng = 1
sai = 4 | đúng = 5


**Sửa:** BFS/DFS để lan truyền.

### Lỗi 4 — Dùng chung mảng `seen` cho mọi lần thử

In [18]:
def wrong_shared_seen(bombs):
    n = len(bombs); g = build_graph(bombs); seen = [False] * n; best = 0     # SAI: không reset
    for s in range(n):
        if seen[s]: continue
        seen[s] = True; q = deque([s]); cnt = 1
        while q:
            u = q.popleft()
            for v in g[u]:
                if not seen[v]: seen[v] = True; cnt += 1; q.append(v)
        best = max(best, cnt)
    return best

rng = random.Random(4); shown = 0
for _ in range(3000):
    bombs = random_bombs(rng.randint(2, 7), rng, 20, 8)
    if wrong_shared_seen(bombs) != max_bombs_bfs(bombs):
        print("phản ví dụ:", bombs, "| sai =", wrong_shared_seen(bombs), "| đúng =", max_bombs_bfs(bombs)); shown += 1
        if shown == 2: break

phản ví dụ: [[10, 2, 4], [17, 18, 6], [9, 6, 2], [9, 7, 1], [9, 9, 4], [6, 10, 5]] | sai = 2 | đúng = 4
phản ví dụ: [[12, 3, 6], [13, 17, 4], [6, 8, 8], [9, 3, 5], [1, 10, 5], [17, 7, 7], [14, 20, 5]] | sai = 2 | đúng = 4


Đây là thói quen từ bài "đếm thành phần liên thông" (vô hướng). Với đồ thị có hướng, một đỉnh đã thăm từ `s₁` vẫn có thể là điểm bắt đầu **tốt hơn** (đến được nhiều đỉnh hơn, kể cả `s₁`). **Sửa:** `seen` mới cho mỗi điểm bắt đầu.

### Lỗi 5 — Đếm `cnt` khi **lấy ra** thay vì khi **đánh dấu**, mà không chặn trùng

Nếu đánh dấu `seen` lúc lấy ra khỏi hàng đợi, một đỉnh có thể được đẩy vào nhiều lần ⇒ đếm trùng. Quy tắc an toàn: **đánh dấu ngay khi đẩy vào** (như mọi cài đặt trong notebook).

In [19]:
def wrong_mark_on_pop(bombs):
    n = len(bombs); g = build_graph(bombs); best = 0
    for s in range(n):
        seen = [False] * n; q = deque([s]); cnt = 0
        while q:
            u = q.popleft(); cnt += 1                           # SAI: u có thể đã được đếm
            seen[u] = True
            for v in g[u]:
                if not seen[v]: q.append(v)
        best = max(best, cnt)
    return best

print("ví dụ 3: sai =", wrong_mark_on_pop(examples[2][0]), "| đúng =", 5)
print("hai quả trùng tâm: sai =", wrong_mark_on_pop([[1, 1, 1], [1, 1, 1]]), "| đúng =", 2)

ví dụ 3: sai = 6 | đúng = 5
hai quả trùng tâm: sai = 2 | đúng = 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Quan hệ "A ảnh hưởng B" **không đối xứng** ⇒ đồ thị **có hướng**; đừng dùng Union-Find.
- "Chọn một điểm khởi đầu, tối đa số đỉnh lan tới" với `n` nhỏ ⇒ BFS/DFS từ **mọi** đỉnh, `O(n · (n + E))`.
- Hình học với tọa độ nguyên ⇒ so sánh **bình phương**, tránh `sqrt`.
- `n` vừa (≤ vài nghìn) và cần mọi tập đến được ⇒ bitset closure.
- Họ bài: 547 *Number of Provinces* (vô hướng — so sánh), 841 *Keys and Rooms*, 1306 *Jump Game III*, 1971 *Find if Path Exists in Graph*, 2492 *Minimum Score of a Path Between Two Cities*.

## 15. Biến thể của bài

1. **Được kích nổ `k` quả** để tối đa tổng số nổ: bài phủ tập (NP-khó tổng quát); với DAG thành phần liên thông mạnh có thể tham lam/heuristic.
2. **Số quả tối thiểu cần kích nổ để nổ hết**: = số thành phần liên thông mạnh **không có cạnh vào** trong đồ thị nén (Bài 5).
3. **Bom nổ có độ trễ** (thời gian lan = khoảng cách): Dijkstra để tính thời điểm mỗi quả nổ.
4. **Nhiều truy vấn "kích quả `s`"**: bitset closure trả lời `O(1)` mỗi truy vấn.

In [20]:
def explosion_times(bombs, s):
    """Biến thể 3: lan với tốc độ 1 đơn vị khoảng cách/giây, trả về thời điểm nổ (float) của mỗi quả, None nếu không nổ."""
    import heapq
    n = len(bombs); g = build_graph(bombs); INF = float("inf"); t = [INF] * n; t[s] = 0.0; pq = [(0.0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > t[u]: continue
        for v in g[u]:
            w = math.dist(bombs[u][:2], bombs[v][:2])
            if d + w < t[v]: t[v] = d + w; heapq.heappush(pq, (d + w, v))
    return [None if x == INF else round(x, 3) for x in t]

print("thời điểm nổ khi kích bom 0 (ví dụ 3):", explosion_times(examples[2][0], 0))

thời điểm nổ khi kích bom 0 (ví dụ 3): [0.0, 1.414, 2.828, 4.243, 5.657]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng thuần | quét lặp | `O(n⁴)` | `O(n)` | kiểm chứng |
| 2. Đồ thị + BFS | đỉnh đến được | `O(n³)` | `O(n²)` | **nộp bài** |
| 3. DFS + dừng sớm | như trên | `O(n³)` | `O(n²)` | thường nhanh hơn |
| 4. Bitset closure | Floyd–Warshall bit | `O(n³/64)` | `O(n²/64)` | cần mọi tập đến được |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1,1,2],[3,1,2],[5,1,2]]`? `[[1,1,1],[3,1,1]]`? `[[1,1,10],[2,2,1],[30,30,100]]`?

**Bài 2 (tính tay):** `[[0,0,5],[3,4,1],[10,0,6],[6,8,2]]` (tọa độ có thể là 0 trong bài tập này). Liệt kê các cạnh và đáp án.

**Bài 3 (điền TODO):** hoàn thiện `edge_list(bombs)` trả về danh sách cặp `(i, j)` mà bom `i` kích nổ trực tiếp bom `j`.

**Bài 4 (tự viết lại):** viết `best_starts(bombs)` trả về **mọi** chỉ số bom mà kích nó cho số nổ lớn nhất.

**Bài 5 (mini challenge):** số quả **ít nhất** cần kích nổ (đồng thời) để mọi quả đều nổ. Dùng thành phần liên thông mạnh (Kosaraju/Tarjan lặp) và đếm thành phần không có cạnh vào; kiểm chứng bằng brute force tập con với `n ≤ 10`. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def edge_list(bombs):
    # TODO: hai vòng for, dùng reaches(bombs[i], bombs[j]) với i != j
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Ba quả cách nhau 2, bán kính 2 ⇒ mọi cặp kề kích nổ nhau ⇒ `3`. (2) Cách 2, bán kính 1 ⇒ `1`. (3) Quả 0 với tới quả 1; quả 2 bán kính 100 ở `(30,30)` cách quả 0 khoảng `41` ⇒ với tới cả hai ⇒ `3`.

**Bài 2:** (1) Tính `d²` từng cặp: `0-1`: 25, `0-2`: 100, `0-3`: 100, `1-3`: 25, `2-3`: 80. (2) Cạnh: `0→1` (25 ≤ 25), `2→0`? `100 ≤ 36`? không; `2→3`? `80 ≤ 36`? không. (3) Đáp án `2`.

**Bài 3:** (1) `[(i, j) for i ... for j ... if i != j and reaches(...)]`. (2) Cẩn thận chiều. (3) Số cạnh tối đa `n(n−1)`.

**Bài 4:** (1) Tính `cnt[s]` cho mọi `s`. (2) `m = max(cnt)`. (3) Trả `[s for s if cnt[s] == m]`.

**Bài 5:** (1) Nén SCC. (2) Thành phần không có cạnh vào phải được kích ít nhất một quả; một quả mỗi thành phần như vậy là đủ. (3) Brute force: thử mọi tập con theo kích thước tăng dần.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert [Solution().maximumDetonation(b) for b in ([[1, 1, 2], [3, 1, 2], [5, 1, 2]], [[1, 1, 1], [3, 1, 1]], [[1, 1, 10], [2, 2, 1], [30, 30, 100]])] == [3, 1, 3]
# Bài 2
b2 = [[0, 0, 5], [3, 4, 1], [10, 0, 6], [6, 8, 2]]
print("Bài 2 cạnh:"); trace_edges(b2)
assert Solution().maximumDetonation(b2) == 2

# Bài 3
def edge_list_ans(bombs):
    return [(i, j) for i in range(len(bombs)) for j in range(len(bombs)) if i != j and reaches(bombs[i], bombs[j])]

# Bài 4
def best_starts_ans(bombs):
    g = build_graph(bombs); cnt = [count_reachable(g, s) for s in range(len(bombs))]; m = max(cnt)
    return [s for s in range(len(bombs)) if cnt[s] == m]

rng = random.Random(41)
for _ in range(300):
    bombs = random_bombs(rng.randint(1, 10), rng, 30, 10)
    E = edge_list_ans(bombs); g = build_graph(bombs)
    assert sorted(E) == sorted((i, j) for i in range(len(g)) for j in g[i])
    bs = best_starts_ans(bombs)
    assert bs and all(count_reachable(g, s) == max_bombs_bfs(bombs) for s in bs)
print("Bài 4 ví dụ 3:", best_starts_ans(examples[2][0]), "| ví dụ 1:", best_starts_ans(examples[0][0]))
print("Bài 1-4: OK")

Bài 2 cạnh:
  0 -> 1: d² = 25 <= r² = 25
Bài 4 ví dụ 3: [0] | ví dụ 1: [1]
Bài 1-4: OK


In [23]:
# Bài 5 — số quả ít nhất để nổ hết: đếm SCC nguồn (Kosaraju lặp)
from itertools import combinations

def min_initial_bombs(bombs):
    n = len(bombs); g = build_graph(bombs); rg = [[] for _ in range(n)]
    for u in range(n):
        for v in g[u]: rg[v].append(u)
    order, seen = [], [False] * n
    for s in range(n):                                   # lượt 1: thứ tự hoàn thành (DFS lặp)
        if seen[s]: continue
        seen[s] = True; st = [(s, 0)]
        while st:
            u, i = st.pop()
            if i < len(g[u]):
                st.append((u, i + 1)); v = g[u][i]
                if not seen[v]: seen[v] = True; st.append((v, 0))
            else:
                order.append(u)
    comp = [-1] * n; c = 0
    for s in reversed(order):                            # lượt 2 trên đồ thị ngược
        if comp[s] != -1: continue
        comp[s] = c; st = [s]
        while st:
            u = st.pop()
            for v in rg[u]:
                if comp[v] == -1: comp[v] = c; st.append(v)
        c += 1
    has_in = [False] * c
    for u in range(n):
        for v in g[u]:
            if comp[u] != comp[v]: has_in[comp[v]] = True
    return sum(1 for x in has_in if not x)

def min_initial_brute(bombs):
    n = len(bombs); g = build_graph(bombs)
    for k in range(1, n + 1):
        for S in combinations(range(n), k):
            seen = set(S); q = deque(S)
            while q:
                u = q.popleft()
                for v in g[u]:
                    if v not in seen: seen.add(v); q.append(v)
            if len(seen) == n: return k

rng = random.Random(55)
for _ in range(300):
    bombs = random_bombs(rng.randint(1, 9), rng, 30, 10)
    assert min_initial_bombs(bombs) == min_initial_brute(bombs)
print("Bài 5: SCC nguồn khớp brute force; ví dụ 1 ->", min_initial_bombs(examples[0][0]),
      "| ví dụ 2 ->", min_initial_bombs(examples[1][0]), "| ví dụ 3 ->", min_initial_bombs(examples[2][0]))

Bài 5: SCC nguồn khớp brute force; ví dụ 1 -> 1 | ví dụ 2 -> 2 | ví dụ 3 -> 1


## 20. Template nhớ nhanh + Checklist

```python
# Đồ thị có hướng i -> j nếu j trong vùng của i; đếm đỉnh đến được từ mỗi đỉnh
g = [[j for j in range(n) if j != i and
      (x[i]-x[j])**2 + (y[i]-y[j])**2 <= r[i]**2] for i in range(n)]
best = 0
for s in range(n):
    seen = [False] * n; seen[s] = True; q = deque([s]); cnt = 1
    while q:
        u = q.popleft()
        for v in g[u]:
            if not seen[v]: seen[v] = True; cnt += 1; q.append(v)
    best = max(best, cnt)
```

**Checklist trước khi submit:**
- [ ] Đồ thị **có hướng** (bán kính của bom nguồn).
- [ ] So sánh `dx² + dy² ≤ r²` (gồm biên, số nguyên).
- [ ] `seen` mới cho mỗi điểm bắt đầu.
- [ ] Đánh dấu khi đẩy vào hàng đợi.
- [ ] Dừng sớm khi đạt `n` (tùy chọn).